In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Surveiller ou annuler un job

Consulte la liste de tes workloads sur la [page Workloads](https://quantum.cloud.ibm.com/workloads).

## Voir le statut d'un job
Accède à ton [tableau Workloads](https://quantum.cloud.ibm.com/workloads) et vérifie dans la colonne Status si un job s'est terminé ou a échoué.

## Voir l'utilisation restante
Accède à ton [tableau Instances](https://quantum.cloud.ibm.com/instances) et sélectionne l'onglet correspondant au plan dont tu veux consulter l'utilisation restante. Le temps total utilisé et le temps total restant sur ton plan sont affichés.

## Voir les métriques sur le nombre de jobs et workloads soumis
Accède à la [page Analytics](https://quantum.cloud.ibm.com/analytics) pour voir le nombre total de jobs soumis, ainsi qu'un décompte des workloads batch et des workloads de session. Note que tu ne peux voir la page Analytics que pour les comptes que tu possèdes ou gères.

## Surveiller un job
Utilise l'instance de job pour vérifier le statut du job ou récupérer les résultats en appelant la commande appropriée :

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | Consulte les résultats du job immédiatement après son achèvement. Les résultats sont disponibles une fois le job terminé. Par conséquent, job.result() est un appel bloquant jusqu'à la fin du job.         |
| job.job\_id()                 | Retourne l'identifiant unique du job. Récupérer les résultats du job ultérieurement nécessite l'identifiant du job. Il est donc recommandé de sauvegarder les identifiants des jobs que tu pourrais vouloir récupérer plus tard. |
| job.status()                  | Vérifie le statut du job.                                                                                                                                                                                    |
| job = service.job(\<job\_id>) | Récupère un job que tu as soumis précédemment. Cet appel nécessite l'identifiant du job.                                                                                                                     |

<span id="retrieve-later"></span>
## Récupérer les résultats d'un job ultérieurement
Appelle `service.job(\<job\_id>)` pour récupérer un job que tu as soumis précédemment. Si tu n'as pas l'identifiant du job, ou si tu veux récupérer plusieurs jobs à la fois — y compris des jobs provenant de QPUs (unités de traitement quantique) retirées —, appelle plutôt `service.jobs()` avec des filtres optionnels. Voir [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` retourne également les jobs exécutés depuis le package déprécié `qiskit-ibm-provider`. Les jobs soumis par l'ancien package (également déprécié) `qiskit-ibmq-provider` ne sont plus disponibles.

### Exemple
Cet exemple retourne les 10 jobs runtime les plus récents qui ont été exécutés sur `my_backend` :

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>